In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%pip install -q -U ultralytics "lap>=0.5.12" pandas tqdm

import subprocess
from pathlib import Path

TRACKEVAL_PATH = Path("/kaggle/working/TrackEval")
if not TRACKEVAL_PATH.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/JonathonLuiten/TrackEval.git",
            str(TRACKEVAL_PATH),
        ],
        check=True,
    )

In [ ]:
import sys
import json
import gc
import time
import shutil
import configparser

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
from tqdm.auto import tqdm
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/mot20_ocsort_benchmark")
OUT.mkdir(parents=True, exist_ok=True)

MODELS = {
    "yolo26s": "yolo26s.pt",
    "yolo26m": "yolo26m.pt",
}

TRACKERS = ["ocsort"]
# For a fresh, controlled comparison of both trackers, use:
# TRACKERS = ["bytetrack", "ocsort"]

IMGSZ = 1280
CONF = 0.10
IOU = 0.70
MAX_DET = 1000                # Important for crowded MOT20 frames
PRECISION = 32                # FP32; consistent across every run
WARMUP_FRAMES = 10

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Kaggle: Settings → Accelerator → GPU."
    )
DEVICE = 0

# Locate training sequences containing images.
sequence_dirs = sorted({
    p.parent
    for p in INPUT.rglob("img1")
    if p.is_dir()
    and p.parent.name.startswith("MOT20-")
    and p.parent.parent.name == "train"
})

sequence_dirs = [
    p for p in sequence_dirs if any((p / "img1").glob("*.jpg"))
]

if not sequence_dirs:
    raise FileNotFoundError("No MOT20 train image sequences found.")

SEQ_DIRS = {}
for path in sequence_dirs:
    if path.name in SEQ_DIRS:
        raise RuntimeError(f"Duplicate image sequence found: {path.name}")
    SEQ_DIRS[path.name] = path

SEQUENCES = sorted(SEQ_DIRS)

# Find GT in either the image dataset or separate labels dataset.
gt_candidates = list(INPUT.rglob("gt.txt"))
GT_FILES = {}
FRAME_PATHS = {}
SEQ_LENGTHS = {}
SEQ_FPS = {}

for seq in SEQUENCES:
    candidates = [
        p for p in gt_candidates
        if p.parent.name == "gt"
        and p.parent.parent.name == seq
        and p.parent.parent.parent.name == "train"
    ]
    if not candidates:
        raise FileNotFoundError(f"No train/{seq}/gt/gt.txt found.")

    if len(candidates) > 1:
        contents = {p.read_bytes() for p in candidates}
        if len(contents) != 1:
            raise RuntimeError(f"Conflicting GT files for {seq}: {candidates}")
    GT_FILES[seq] = candidates[0]

    paths = sorted(
        (SEQ_DIRS[seq] / "img1").glob("*.jpg"),
        key=lambda p: int(p.stem),
    )
    frame_numbers = [int(p.stem) for p in paths]

    # Full, consecutive frames are required for this evaluation.
    if frame_numbers != list(range(1, len(paths) + 1)):
        raise RuntimeError(f"{seq} has missing or non-consecutive frames.")

    FRAME_PATHS[seq] = paths
    SEQ_LENGTHS[seq] = len(paths)

    info = configparser.ConfigParser()
    info.read(SEQ_DIRS[seq] / "seqinfo.ini")
    SEQ_FPS[seq] = float(info["Sequence"]["frameRate"])

# Check that the installed package includes the requested trackers.
TRACKER_FILES = {}
for tracker in TRACKERS:
    path = (
        Path(ultralytics.__file__).parent
        / "cfg" / "trackers" / f"{tracker}.yaml"
    )
    if not path.exists():
        raise RuntimeError(
            f"{path.name} is missing. Upgrade Ultralytics, restart "
            "the session, then rerun these cells."
        )
    TRACKER_FILES[tracker] = path
    shutil.copy2(path, OUT / path.name)

manifest = {
    "ultralytics_version": ultralytics.__version__,
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(DEVICE),
    "models": MODELS,
    "trackers": TRACKERS,
    "imgsz": IMGSZ,
    "conf": CONF,
    "iou": IOU,
    "max_det": MAX_DET,
    "precision": PRECISION,
    "sequence_lengths": SEQ_LENGTHS,
}
(OUT / "configuration.json").write_text(json.dumps(manifest, indent=2))

print(json.dumps(manifest, indent=2))
for seq in SEQUENCES:
    print(seq, "GT:", GT_FILES[seq])

In [ ]:
def synchronize():
    torch.cuda.synchronize(DEVICE)

system_rows = []

for tracker_name in TRACKERS:
    for model_name, weights in MODELS.items():
        run_name = f"{model_name}_{tracker_name}"
        run_dir = OUT / "predictions" / run_name / "data"
        run_dir.mkdir(parents=True, exist_ok=True)

        for seq in SEQUENCES:
            gc.collect()
            torch.cuda.empty_cache()

            # Fresh model/predictor gives this sequence a fresh tracker.
            model = YOLO(weights)
            frames = FRAME_PATHS[seq]

            first_frame = cv2.imread(str(frames[0]))
            if first_frame is None:
                raise RuntimeError(f"Cannot read {frames[0]}")

            common_args = dict(
                classes=[0],          # Person
                imgsz=IMGSZ,
                conf=CONF,
                iou=IOU,
                max_det=MAX_DET,
                device=DEVICE,
                quantize=PRECISION,
                verbose=False,
                save=False,
            )

            # Warm up the detector without creating/advancing tracks.
            for _ in range(WARMUP_FRAMES):
                warmup_result = model.predict(
                    first_frame, **common_args
                )
            del warmup_result
            synchronize()
            torch.cuda.reset_peak_memory_stats(DEVICE)

            timing_rows = []
            prediction_path = run_dir / f"{seq}.txt"
            sequence_start = time.perf_counter()

            with prediction_path.open("w") as output:
                for frame_path in tqdm(
                    frames, desc=f"{run_name} | {seq}"
                ):
                    frame_number = int(frame_path.stem)

                    synchronize()
                    start = time.perf_counter()

                    frame = cv2.imread(str(frame_path))
                    if frame is None:
                        raise RuntimeError(f"Cannot read {frame_path}")

                    result = model.track(
                        frame,
                        persist=True,
                        tracker=str(TRACKER_FILES[tracker_name]),
                        **common_args,
                    )[0]

                    boxes = result.boxes
                    if (
                        boxes is not None
                        and len(boxes) > 0
                        and boxes.id is not None
                    ):
                        coordinates = boxes.xyxy.cpu().numpy()
                        identities = boxes.id.int().cpu().numpy()
                        scores = boxes.conf.cpu().numpy()
                    else:
                        coordinates = np.empty((0, 4))
                        identities = np.empty(0, dtype=int)
                        scores = np.empty(0)

                    synchronize()
                    elapsed_ms = (time.perf_counter() - start) * 1000

                    timing_rows.append({
                        "frame": frame_number,
                        "latency_ms": elapsed_ms,
                        "active_tracks": len(identities),
                    })

                    # MOTChallenge tracking output.
                    for box, track_id, score in zip(
                        coordinates, identities, scores
                    ):
                        x1, y1, x2, y2 = map(float, box)
                        output.write(
                            f"{frame_number},{int(track_id)},"
                            f"{x1:.3f},{y1:.3f},"
                            f"{x2-x1:.3f},{y2-y1:.3f},"
                            f"{float(score):.6f},-1,-1,-1\n"
                        )

            sequence_wall_s = time.perf_counter() - sequence_start
            timings = pd.DataFrame(timing_rows)
            latency = timings["latency_ms"].to_numpy()
            processing_s = float(latency.sum() / 1000)

            record = {
                "run": run_name,
                "model": model_name,
                "tracker": tracker_name,
                "sequence": seq,
                "frames": len(frames),
                "fps": len(frames) / processing_s,
                "mean_latency_ms": float(latency.mean()),
                "p50_latency_ms": float(np.percentile(latency, 50)),
                "p95_latency_ms": float(np.percentile(latency, 95)),
                "processing_time_s": processing_s,
                "sequence_wall_time_s": sequence_wall_s,
                "peak_gpu_allocated_mb": (
                    torch.cuda.max_memory_allocated(DEVICE) / 1024**2
                ),
                "peak_gpu_reserved_mb": (
                    torch.cuda.max_memory_reserved(DEVICE) / 1024**2
                ),
            }
            system_rows.append(record)

            timings.to_csv(
                run_dir.parent / f"{seq}_timings.csv", index=False
            )

            # Save completed sequences immediately.
            pd.DataFrame(system_rows).to_csv(
                OUT / "system_per_sequence.csv", index=False
            )
            print(record)

            del result, model
            gc.collect()
            torch.cuda.empty_cache()

system_df = pd.DataFrame(system_rows)
display(system_df.round(3))

In [ ]:
# Repair deprecated NumPy aliases in TrackEval.
from pathlib import Path
import re
import sys
import importlib

trackeval_dir = Path("/kaggle/working/TrackEval/trackeval")
if not trackeval_dir.is_dir():
    raise FileNotFoundError(trackeval_dir)

replacements = {
    "float": "float",
    "int": "int",
    "bool": "bool",
    "complex": "complex",
    "object": "object",
    "str": "str",
}

pattern = re.compile(
    r"\bnp\.(float|int|bool|complex|object|str)\b"
)

changed = 0
for file in trackeval_dir.rglob("*.py"):
    original = file.read_text()
    updated = pattern.sub(
        lambda match: replacements[match.group(1)],
        original,
    )
    if updated != original:
        file.write_text(updated)
        changed += 1

# Clear cached imports so Cell 4 loads the repaired code.
for module_name in list(sys.modules):
    if module_name == "trackeval" or module_name.startswith("trackeval."):
        del sys.modules[module_name]

importlib.invalidate_caches()

print(f"Repaired {changed} TrackEval files.")
print("Now rerun Cell 4, followed by Cell 5.")

In [ ]:
sys.path.insert(0, str(TRACKEVAL_PATH))
import trackeval

TE_GT = OUT / "trackeval_gt"
TE_GT.mkdir(exist_ok=True)

for seq in SEQUENCES:
    target = TE_GT / seq / "gt"
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(GT_FILES[seq], target / "gt.txt")

RUN_NAMES = [
    f"{model}_{tracker}"
    for tracker in TRACKERS
    for model in MODELS
]

eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({
    "USE_PARALLEL": False,
    "PRINT_RESULTS": True,
    "PRINT_CONFIG": False,
    "OUTPUT_SUMMARY": True,
    "OUTPUT_DETAILED": True,
    "PLOT_CURVES": False,
    "BREAK_ON_ERROR": True,
})

dataset_config = (
    trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
)
dataset_config.update({
    "GT_FOLDER": str(TE_GT),
    "TRACKERS_FOLDER": str(OUT / "predictions"),
    "OUTPUT_FOLDER": str(OUT / "tracking_evaluation"),
    "TRACKERS_TO_EVAL": RUN_NAMES,
    "CLASSES_TO_EVAL": ["pedestrian"],
    "BENCHMARK": "MOT20",
    "SPLIT_TO_EVAL": "train",
    "SKIP_SPLIT_FOL": True,
    "TRACKER_SUB_FOLDER": "data",
    "SEQ_INFO": SEQ_LENGTHS,
    "GT_LOC_FORMAT": "{gt_folder}/{seq}/gt/gt.txt",
    "DO_PREPROC": True,
    "PRINT_CONFIG": False,
})

dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
metrics = [
    trackeval.metrics.HOTA(),
    trackeval.metrics.CLEAR(),
    trackeval.metrics.Identity(),
]

evaluation_results, evaluation_messages = (
    trackeval.Evaluator(eval_config).evaluate([dataset], metrics)
)

tracking_rows = []

for run_name in RUN_NAMES:
    combined = evaluation_results[dataset.get_name()][run_name][
        "COMBINED_SEQ"
    ]["pedestrian"]

    hota = combined["HOTA"]
    clear = combined["CLEAR"]
    identity = combined["Identity"]

    tracking_rows.append({
        "run": run_name,
        "HOTA": float(np.mean(hota["HOTA"]) * 100),
        "IDF1": float(identity["IDF1"] * 100),
        "MOTA": float(clear["MOTA"] * 100),
        "ID_switches": int(clear["IDSW"]),
        "fragmentations": int(clear["Frag"]),
        "false_positives": int(clear["CLR_FP"]),
        "false_negatives": int(clear["CLR_FN"]),
        "tracking_precision": float(clear["CLR_Pr"] * 100),
        "tracking_recall": float(clear["CLR_Re"] * 100),
    })

tracking_df = pd.DataFrame(tracking_rows)
tracking_df.to_csv(OUT / "tracking_summary.csv", index=False)
display(tracking_df.round(3))

In [ ]:
system_summary_rows = []

for run_name in RUN_NAMES:
    subset = system_df[system_df["run"] == run_name]

    timing_arrays = [
        pd.read_csv(
            OUT / "predictions" / run_name / f"{seq}_timings.csv"
        )["latency_ms"].to_numpy()
        for seq in SEQUENCES
    ]
    latency = np.concatenate(timing_arrays)

    system_summary_rows.append({
        "run": run_name,
        "FPS": len(latency) / (latency.sum() / 1000),
        "mean_latency_ms": float(latency.mean()),
        "P50_latency_ms": float(np.percentile(latency, 50)),
        "P95_latency_ms": float(np.percentile(latency, 95)),
        "peak_GPU_allocated_MB": subset["peak_gpu_allocated_mb"].max(),
        "peak_GPU_reserved_MB": subset["peak_gpu_reserved_mb"].max(),
        "total_processing_s": float(latency.sum() / 1000),
        "total_sequence_wall_s": subset["sequence_wall_time_s"].sum(),
    })

summary = tracking_df.merge(
    pd.DataFrame(system_summary_rows), on="run"
)
summary["R@1"] = np.nan
summary["R@5"] = np.nan

summary.to_csv(OUT / "benchmark_summary.csv", index=False)
display(summary.round(3))
print("Results:", OUT / "benchmark_summary.csv")

In [ ]:
import subprocess
from IPython.display import Video, display

VIS_RUN = "yolo26m_ocsort"       # Or "yolo26s_ocsort"
VIS_SEQ = "MOT20-05"
PREVIEW_FRAMES = 300            # Set to None for the complete sequence
VIDEO_WIDTH = 1280

prediction_file = OUT / "predictions" / VIS_RUN / "data" / f"{VIS_SEQ}.txt"

tracks = {}
for line in prediction_file.read_text().splitlines():
    if line.strip():
        row = [float(v) for v in line.split(",")]
        tracks.setdefault(int(row[0]), []).append(row)

paths = FRAME_PATHS[VIS_SEQ]
if PREVIEW_FRAMES is not None:
    paths = paths[:PREVIEW_FRAMES]

first = cv2.imread(str(paths[0]))
if first is None:
    raise RuntimeError(f"Cannot read {paths[0]}")

height, width = first.shape[:2]
video_width = VIDEO_WIDTH - VIDEO_WIDTH % 2
video_height = int(height * video_width / width)
video_height -= video_height % 2
scale_x = video_width / width
scale_y = video_height / height

video_dir = OUT / "videos"
video_dir.mkdir(exist_ok=True)
raw_video = video_dir / f"{VIS_RUN}_{VIS_SEQ}_raw.mp4"
final_video = video_dir / f"{VIS_RUN}_{VIS_SEQ}.mp4"

writer = cv2.VideoWriter(
    str(raw_video),
    cv2.VideoWriter_fourcc(*"mp4v"),
    SEQ_FPS[VIS_SEQ],
    (video_width, video_height),
)
if not writer.isOpened():
    raise RuntimeError("Could not open video writer.")

try:
    for path in tqdm(paths, desc="Rendering tracking video"):
        frame = cv2.imread(str(path))
        if frame is None:
            raise RuntimeError(f"Cannot read {path}")
        frame = cv2.resize(frame, (video_width, video_height))
        frame_number = int(path.stem)

        for row in tracks.get(frame_number, []):
            track_id = int(row[1])
            x, y, w, h = row[2:6]
            color = tuple(
                int(v)
                for v in np.random.default_rng(track_id).integers(
                    70, 256, size=3
                )
            )

            p1 = (round(x * scale_x), round(y * scale_y))
            p2 = (round((x + w) * scale_x), round((y + h) * scale_y))

            cv2.rectangle(frame, p1, p2, color, 2)
            cv2.putText(
                frame, f"ID {track_id} | {row[6]:.2f}",
                (max(0, p1[0]), max(20, p1[1] - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5, color, 2, cv2.LINE_AA,
            )

        cv2.putText(
            frame, f"{VIS_RUN} | frame {frame_number}",
            (15, 30), cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (255, 255, 255), 2, cv2.LINE_AA,
        )
        writer.write(frame)
finally:
    writer.release()

# Convert to browser-compatible H.264.
if shutil.which("ffmpeg"):
    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(raw_video),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(final_video),
        ],
        check=True,
    )
    display(Video(str(final_video), embed=True, width=960))
else:
    print("Video saved:", raw_video)
    print("Download it from Kaggle Output to watch it.")

# *DEEEP OCSORT*

In [ ]:
%pip install -q -U ultralytics "lap>=0.5.12" pandas tqdm pyyaml scipy

from pathlib import Path
import subprocess
import re
import sys
import importlib

TRACKEVAL_PATH = Path("/kaggle/working/TrackEval")

if not TRACKEVAL_PATH.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/JonathonLuiten/TrackEval.git",
            str(TRACKEVAL_PATH),
        ],
        check=True,
    )

# Repair removed NumPy aliases used by older TrackEval code.
alias_pattern = re.compile(
    r"\bnp\.(float|int|bool|complex|object|str)\b"
)

changed = 0
for path in (TRACKEVAL_PATH / "trackeval").rglob("*.py"):
    original = path.read_text()
    updated = alias_pattern.sub(lambda m: m.group(1), original)
    if updated != original:
        path.write_text(updated)
        changed += 1

for name in list(sys.modules):
    if name == "trackeval" or name.startswith("trackeval."):
        del sys.modules[name]

importlib.invalidate_caches()
print(f"TrackEval ready. Repaired {changed} files.")

In [ ]:
import json
import gc
import time
import shutil
import configparser

import cv2
import yaml
import numpy as np
import pandas as pd
import torch
import ultralytics

from tqdm.auto import tqdm
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/mot20_deepocsort_benchmark")
OUT.mkdir(parents=True, exist_ok=True)

MODELS = {
    "yolo26s": "yolo26s.pt",
    "yolo26m": "yolo26m.pt",
}

IMGSZ = 1280
CONF = 0.10
IOU = 0.70
MAX_DET = 1000
PRECISION = 32
WARMUP_FRAMES = 10

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle Settings → Accelerator.")

DEVICE = 0

# Build Deep OC-SORT configuration with ReID enabled.
default_tracker = (
    Path(ultralytics.__file__).parent
    / "cfg" / "trackers" / "deepocsort.yaml"
)

if not default_tracker.exists():
    raise FileNotFoundError(
        "deepocsort.yaml is missing. Upgrade Ultralytics, restart "
        "the session, and rerun Cell 2."
    )

tracker_config = yaml.safe_load(default_tracker.read_text())
tracker_config["with_reid"] = True
tracker_config["model"] = "auto"

TRACKER_FILE = OUT / "deepocsort_reid.yaml"
TRACKER_FILE.write_text(
    yaml.safe_dump(tracker_config, sort_keys=False)
)

# Find MOT20 training image sequences.
sequence_dirs = sorted({
    p.parent
    for p in INPUT.rglob("img1")
    if p.is_dir()
    and p.parent.name.startswith("MOT20-")
    and p.parent.parent.name == "train"
    and any(p.glob("*.jpg"))
})

if not sequence_dirs:
    raise FileNotFoundError("No MOT20 train image sequences found.")

SEQ_DIRS = {}
for path in sequence_dirs:
    if path.name in SEQ_DIRS:
        raise RuntimeError(f"Duplicate image sequence: {path.name}")
    SEQ_DIRS[path.name] = path

SEQUENCES = sorted(SEQ_DIRS)
gt_candidates = list(INPUT.rglob("gt.txt"))

GT_FILES = {}
FRAME_PATHS = {}
SEQ_LENGTHS = {}
SEQ_FPS = {}

for seq in SEQUENCES:
    candidates = [
        p for p in gt_candidates
        if p.parent.name == "gt"
        and p.parent.parent.name == seq
        and p.parent.parent.parent.name == "train"
    ]

    if not candidates:
        raise FileNotFoundError(f"No train/{seq}/gt/gt.txt found.")

    if len(candidates) > 1:
        if len({p.read_bytes() for p in candidates}) != 1:
            raise RuntimeError(f"Conflicting GT files: {candidates}")

    GT_FILES[seq] = candidates[0]

    frames = sorted(
        (SEQ_DIRS[seq] / "img1").glob("*.jpg"),
        key=lambda p: int(p.stem),
    )
    numbers = [int(p.stem) for p in frames]

    if numbers != list(range(1, len(frames) + 1)):
        raise RuntimeError(f"{seq} has missing/non-consecutive frames.")

    info_path = SEQ_DIRS[seq] / "seqinfo.ini"
    if not info_path.exists():
        info_path = GT_FILES[seq].parent.parent / "seqinfo.ini"

    info = configparser.ConfigParser()
    info.read(info_path)
    if not info.has_section("Sequence"):
        raise FileNotFoundError(f"Cannot read seqinfo.ini for {seq}")

    FRAME_PATHS[seq] = frames
    SEQ_LENGTHS[seq] = len(frames)
    SEQ_FPS[seq] = float(info["Sequence"]["frameRate"])

manifest = {
    "ultralytics_version": ultralytics.__version__,
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(DEVICE),
    "models": MODELS,
    "tracker": "deepocsort",
    "tracker_config": tracker_config,
    "imgsz": IMGSZ,
    "conf": CONF,
    "iou": IOU,
    "max_det": MAX_DET,
    "precision": PRECISION,
    "sequence_lengths": SEQ_LENGTHS,
}

(OUT / "configuration.json").write_text(
    json.dumps(manifest, indent=2)
)

print(json.dumps(manifest, indent=2))
for seq in SEQUENCES:
    print(seq, "GT:", GT_FILES[seq])

In [ ]:
def synchronize():
    torch.cuda.synchronize(DEVICE)

system_rows = []

for model_name, weights in MODELS.items():
    run_name = f"{model_name}_deepocsort"
    run_dir = OUT / "predictions" / run_name / "data"
    run_dir.mkdir(parents=True, exist_ok=True)

    for seq in SEQUENCES:
        gc.collect()
        torch.cuda.empty_cache()

        model = YOLO(weights)
        frames = FRAME_PATHS[seq]

        common_args = {
            "classes": [0],
            "imgsz": IMGSZ,
            "conf": CONF,
            "iou": IOU,
            "max_det": MAX_DET,
            "device": DEVICE,
            "quantize": PRECISION,
            "verbose": False,
            "save": False,
        }

        first_frame = cv2.imread(str(frames[0]))
        if first_frame is None:
            raise RuntimeError(f"Cannot read {frames[0]}")

        # Detector-only warm-up leaves tracker state untouched.
        for _ in range(WARMUP_FRAMES):
            warmup_result = model.predict(first_frame, **common_args)
        del warmup_result
        synchronize()
        torch.cuda.reset_peak_memory_stats(DEVICE)

        timing_rows = []
        prediction_path = run_dir / f"{seq}.txt"
        sequence_start = time.perf_counter()

        with prediction_path.open("w") as output:
            for frame_path in tqdm(
                frames, desc=f"{run_name} | {seq}"
            ):
                frame_number = int(frame_path.stem)

                synchronize()
                start = time.perf_counter()

                frame = cv2.imread(str(frame_path))
                if frame is None:
                    raise RuntimeError(f"Cannot read {frame_path}")

                result = model.track(
                    frame,
                    persist=True,
                    tracker=str(TRACKER_FILE),
                    **common_args,
                )[0]

                boxes = result.boxes

                if (
                    boxes is not None
                    and len(boxes) > 0
                    and boxes.id is not None
                ):
                    coordinates = boxes.xyxy.cpu().numpy()
                    identities = boxes.id.int().cpu().numpy()
                    scores = boxes.conf.cpu().numpy()
                else:
                    coordinates = np.empty((0, 4))
                    identities = np.empty(0, dtype=int)
                    scores = np.empty(0)

                synchronize()
                latency_ms = (time.perf_counter() - start) * 1000

                timing_rows.append({
                    "frame": frame_number,
                    "latency_ms": latency_ms,
                    "active_tracks": len(identities),
                })

                for box, track_id, score in zip(
                    coordinates, identities, scores
                ):
                    x1, y1, x2, y2 = map(float, box)
                    output.write(
                        f"{frame_number},{int(track_id)},"
                        f"{x1:.3f},{y1:.3f},"
                        f"{x2-x1:.3f},{y2-y1:.3f},"
                        f"{float(score):.6f},-1,-1,-1\n"
                    )

        sequence_wall_s = time.perf_counter() - sequence_start

        timings = pd.DataFrame(timing_rows)
        latency = timings["latency_ms"].to_numpy()
        processing_s = float(latency.sum() / 1000)

        record = {
            "run": run_name,
            "model": model_name,
            "tracker": "deepocsort",
            "sequence": seq,
            "frames": len(frames),
            "fps": len(frames) / processing_s,
            "mean_latency_ms": float(latency.mean()),
            "p50_latency_ms": float(np.percentile(latency, 50)),
            "p95_latency_ms": float(np.percentile(latency, 95)),
            "first_frame_latency_ms": float(latency[0]),
            "processing_time_s": processing_s,
            "sequence_wall_time_s": sequence_wall_s,
            "peak_gpu_allocated_mb": (
                torch.cuda.max_memory_allocated(DEVICE) / 1024**2
            ),
            "peak_gpu_reserved_mb": (
                torch.cuda.max_memory_reserved(DEVICE) / 1024**2
            ),
        }

        system_rows.append(record)

        timings.to_csv(
            run_dir.parent / f"{seq}_timings.csv", index=False
        )
        pd.DataFrame(system_rows).to_csv(
            OUT / "system_per_sequence.csv", index=False
        )

        print(record)

        del model, result, boxes, frame, first_frame
        gc.collect()
        torch.cuda.empty_cache()

system_df = pd.DataFrame(system_rows)
display(system_df.round(3))

In [ ]:
sys.path.insert(0, str(TRACKEVAL_PATH))
import trackeval

TE_GT = OUT / "trackeval_gt"
TE_GT.mkdir(exist_ok=True)

for seq in SEQUENCES:
    target = TE_GT / seq / "gt"
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(GT_FILES[seq], target / "gt.txt")

RUN_NAMES = [f"{model}_deepocsort" for model in MODELS]

eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({
    "USE_PARALLEL": False,
    "PRINT_RESULTS": True,
    "PRINT_CONFIG": False,
    "OUTPUT_SUMMARY": True,
    "OUTPUT_DETAILED": True,
    "PLOT_CURVES": False,
    "BREAK_ON_ERROR": True,
})

dataset_config = (
    trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
)
dataset_config.update({
    "GT_FOLDER": str(TE_GT),
    "TRACKERS_FOLDER": str(OUT / "predictions"),
    "OUTPUT_FOLDER": str(OUT / "tracking_evaluation"),
    "TRACKERS_TO_EVAL": RUN_NAMES,
    "CLASSES_TO_EVAL": ["pedestrian"],
    "BENCHMARK": "MOT20",
    "SPLIT_TO_EVAL": "train",
    "SKIP_SPLIT_FOL": True,
    "TRACKER_SUB_FOLDER": "data",
    "SEQ_INFO": SEQ_LENGTHS,
    "GT_LOC_FORMAT": "{gt_folder}/{seq}/gt/gt.txt",
    "DO_PREPROC": True,
    "PRINT_CONFIG": False,
})

dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
metrics = [
    trackeval.metrics.HOTA(),
    trackeval.metrics.CLEAR(),
    trackeval.metrics.Identity(),
]

evaluation_results, evaluation_messages = (
    trackeval.Evaluator(eval_config).evaluate([dataset], metrics)
)

tracking_rows = []

for run_name in RUN_NAMES:
    combined = evaluation_results[dataset.get_name()][run_name][
        "COMBINED_SEQ"
    ]["pedestrian"]

    hota = combined["HOTA"]
    clear = combined["CLEAR"]
    identity = combined["Identity"]

    tracking_rows.append({
        "run": run_name,
        "HOTA": float(np.mean(hota["HOTA"]) * 100),
        "IDF1": float(identity["IDF1"] * 100),
        "MOTA": float(clear["MOTA"] * 100),
        "ID_switches": int(clear["IDSW"]),
        "fragmentations": int(clear["Frag"]),
        "false_positives": int(clear["CLR_FP"]),
        "false_negatives": int(clear["CLR_FN"]),
        "tracking_precision": float(clear["CLR_Pr"] * 100),
        "tracking_recall": float(clear["CLR_Re"] * 100),
    })

tracking_df = pd.DataFrame(tracking_rows)
tracking_df.to_csv(OUT / "tracking_summary.csv", index=False)
display(tracking_df.round(3))

In [ ]:
system_df = pd.read_csv(OUT / "system_per_sequence.csv")
tracking_df = pd.read_csv(OUT / "tracking_summary.csv")

system_summary_rows = []

for run_name in RUN_NAMES:
    subset = system_df[system_df["run"] == run_name]

    latency = np.concatenate([
        pd.read_csv(
            OUT / "predictions" / run_name / f"{seq}_timings.csv"
        )["latency_ms"].to_numpy()
        for seq in SEQUENCES
    ])

    system_summary_rows.append({
        "run": run_name,
        "FPS": len(latency) / (latency.sum() / 1000),
        "mean_latency_ms": float(latency.mean()),
        "P50_latency_ms": float(np.percentile(latency, 50)),
        "P95_latency_ms": float(np.percentile(latency, 95)),
        "peak_GPU_allocated_MB": subset["peak_gpu_allocated_mb"].max(),
        "peak_GPU_reserved_MB": subset["peak_gpu_reserved_mb"].max(),
        "total_processing_s": float(latency.sum() / 1000),
        "total_sequence_wall_s": subset["sequence_wall_time_s"].sum(),
    })

summary = tracking_df.merge(
    pd.DataFrame(system_summary_rows), on="run"
)

# ReID being enabled does not automatically define a retrieval benchmark.
summary["R@1"] = np.nan
summary["R@5"] = np.nan

summary.to_csv(OUT / "benchmark_summary.csv", index=False)
display(summary.round(3))

print("Saved:", OUT / "benchmark_summary.csv")

In [ ]:
from IPython.display import Video, display

VIS_RUN = "yolo26m_deepocsort"    # Or "yolo26s_deepocsort"
VIS_SEQ = "MOT20-05"
PREVIEW_FRAMES = 300             # None = complete sequence
VIDEO_WIDTH = 1280

prediction_file = (
    OUT / "predictions" / VIS_RUN / "data" / f"{VIS_SEQ}.txt"
)
if not prediction_file.exists():
    raise FileNotFoundError(prediction_file)

tracks = {}
for line in prediction_file.read_text().splitlines():
    if line.strip():
        row = [float(v) for v in line.split(",")]
        tracks.setdefault(int(row[0]), []).append(row)

paths = FRAME_PATHS[VIS_SEQ]
if PREVIEW_FRAMES is not None:
    paths = paths[:PREVIEW_FRAMES]
if not paths:
    raise ValueError("No frames selected.")

first = cv2.imread(str(paths[0]))
if first is None:
    raise RuntimeError(f"Cannot read {paths[0]}")

height, width = first.shape[:2]
video_width = VIDEO_WIDTH - VIDEO_WIDTH % 2
video_height = int(height * video_width / width)
video_height -= video_height % 2

scale_x = video_width / width
scale_y = video_height / height

video_dir = OUT / "videos"
video_dir.mkdir(exist_ok=True)
raw_video = video_dir / f"{VIS_RUN}_{VIS_SEQ}_raw.mp4"
final_video = video_dir / f"{VIS_RUN}_{VIS_SEQ}.mp4"

writer = cv2.VideoWriter(
    str(raw_video),
    cv2.VideoWriter_fourcc(*"mp4v"),
    SEQ_FPS[VIS_SEQ],
    (video_width, video_height),
)
if not writer.isOpened():
    raise RuntimeError("Could not open video writer.")

try:
    for path in tqdm(paths, desc="Rendering tracking video"):
        frame = cv2.imread(str(path))
        if frame is None:
            raise RuntimeError(f"Cannot read {path}")

        frame = cv2.resize(frame, (video_width, video_height))
        frame_number = int(path.stem)
        current_tracks = tracks.get(frame_number, [])

        for row in current_tracks:
            track_id = int(row[1])
            x, y, w, h = row[2:6]

            color = tuple(
                int(v)
                for v in np.random.default_rng(track_id).integers(
                    70, 256, size=3
                )
            )

            p1 = (round(x * scale_x), round(y * scale_y))
            p2 = (round((x + w) * scale_x), round((y + h) * scale_y))

            cv2.rectangle(frame, p1, p2, color, 2)
            cv2.putText(
                frame,
                f"ID {track_id} | {row[6]:.2f}",
                (max(0, p1[0]), max(20, p1[1] - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5, color, 2, cv2.LINE_AA,
            )

        cv2.putText(
            frame,
            f"{VIS_RUN} | frame {frame_number} | "
            f"{len(current_tracks)} tracks",
            (15, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (255, 255, 255), 2, cv2.LINE_AA,
        )

        writer.write(frame)
finally:
    writer.release()

if shutil.which("ffmpeg"):
    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(raw_video),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(final_video),
        ],
        check=True,
    )
    display(Video(str(final_video), embed=True, width=960))
    print("Saved:", final_video)
else:
    print("Download this video from Kaggle Output:", raw_video)

# *BOOST TRACK* 

In [ ]:
%pip install -q -U ultralytics "boxmot==25.0.0" pandas tqdm scipy

from pathlib import Path
import subprocess
import importlib
import re
import sys

TRACKEVAL_PATH = Path("/kaggle/working/TrackEval")

if not TRACKEVAL_PATH.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/JonathonLuiten/TrackEval.git",
            str(TRACKEVAL_PATH),
        ],
        check=True,
    )

# Compatibility repair for older TrackEval NumPy aliases.
pattern = re.compile(r"\bnp\.(float|int|bool|complex|object|str)\b")

for path in (TRACKEVAL_PATH / "trackeval").rglob("*.py"):
    original = path.read_text()
    updated = pattern.sub(lambda m: m.group(1), original)
    if updated != original:
        path.write_text(updated)

for name in list(sys.modules):
    if name == "trackeval" or name.startswith("trackeval."):
        del sys.modules[name]

importlib.invalidate_caches()
print("Installation and TrackEval repair completed.")

In [ ]:
# Cell 2 — BoostTrack configuration and dataset discovery

from pathlib import Path
from importlib.metadata import version
import importlib
import inspect
import json
import gc
import time
import shutil
import configparser

import cv2
import numpy as np
import pandas as pd
import torch
import boxmot

from tqdm.auto import tqdm
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/mot20_boosttrack_benchmark")
OUT.mkdir(parents=True, exist_ok=True)

MODELS = {
    "yolo26s": "yolo26s.pt",
    "yolo26m": "yolo26m.pt",
}

IMGSZ = 1280
CONF = 0.10
IOU = 0.70
MAX_DET = 1000
PRECISION = 32
WARMUP_FRAMES = 10

REID_MODEL = "osnet-x0-25-msmt17"
REID_PRECISION = "fp32"
REID_WEIGHTS = OUT / "osnet_x0_25_msmt17.pt"

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Kaggle Settings → Accelerator."
    )

DEVICE = 0

print("BoxMOT version:", version("boxmot"))
print("BoxMOT location:", boxmot.__file__)

# ------------------------------------------------------------
# Resolve the installed BoostTrack / ReID API.
# ------------------------------------------------------------

ReIDConfig = None

for module_name in (
    "boxmot",
    "boxmot.reid",
    "boxmot.reid.specs",
):
    try:
        module = importlib.import_module(module_name)
    except ImportError:
        continue

    candidate = getattr(module, "ReIDConfig", None)
    if candidate is not None:
        ReIDConfig = candidate
        break

create_tracker = getattr(boxmot, "create_tracker", None)
BoostTrack = getattr(boxmot, "BoostTrack", None)

legacy_parameters = (
    inspect.signature(BoostTrack.__init__).parameters
    if BoostTrack is not None else {}
)

if ReIDConfig is not None and create_tracker is not None:
    TRACKER_API = "ReIDConfig"

    def make_tracker():
        return create_tracker(
            "boosttrack",
            backend="python",
            geometry="aabb",
            per_class=False,
            use_embeddings=True,
            reid=ReIDConfig(
                model=REID_MODEL,
                device="cuda:0",
                precision=REID_PRECISION,
                batch_size=32,
            ),
        )

elif BoostTrack is not None and "reid_weights" in legacy_parameters:
    TRACKER_API = "legacy_reid_weights"

    def make_tracker():
        kwargs = {
            "reid_weights": REID_WEIGHTS,
            "device": torch.device("cuda:0"),
            "half": False,
        }

        # Explicitly enable embeddings where this is configurable.
        if "use_embeddings" in legacy_parameters:
            kwargs["use_embeddings"] = True

        if "per_class" in legacy_parameters:
            kwargs["per_class"] = False

        return BoostTrack(**kwargs)

else:
    factory_signature = (
        str(inspect.signature(create_tracker))
        if create_tracker is not None else "not exported"
    )
    boost_signature = (
        str(inspect.signature(BoostTrack.__init__))
        if BoostTrack is not None else "not exported"
    )

    raise RuntimeError(
        f"Unsupported BoxMOT API in version {version('boxmot')}.\n"
        f"create_tracker: {factory_signature}\n"
        f"BoostTrack: {boost_signature}\n"
        "Send this output so the constructor can be matched exactly."
    )

print("Selected tracker API:", TRACKER_API)

# ------------------------------------------------------------
# Locate MOT20 training images.
# ------------------------------------------------------------

sequence_dirs = sorted({
    p.parent
    for p in INPUT.rglob("img1")
    if p.is_dir()
    and p.parent.name.startswith("MOT20-")
    and p.parent.parent.name == "train"
    and any(p.glob("*.jpg"))
})

if not sequence_dirs:
    raise FileNotFoundError(
        "No MOT20 training image sequences found under /kaggle/input."
    )

SEQ_DIRS = {}

for path in sequence_dirs:
    if path.name in SEQ_DIRS:
        raise RuntimeError(
            f"Duplicate image sequence found: {path.name}"
        )
    SEQ_DIRS[path.name] = path

SEQUENCES = sorted(SEQ_DIRS)
gt_candidates = list(INPUT.rglob("gt.txt"))

GT_FILES = {}
FRAME_PATHS = {}
SEQ_LENGTHS = {}
SEQ_FPS = {}

for seq in SEQUENCES:
    candidates = [
        p for p in gt_candidates
        if p.parent.name == "gt"
        and p.parent.parent.name == seq
        and p.parent.parent.parent.name == "train"
    ]

    if not candidates:
        raise FileNotFoundError(
            f"No train/{seq}/gt/gt.txt found."
        )

    if len(candidates) > 1:
        if len({p.read_bytes() for p in candidates}) != 1:
            raise RuntimeError(
                f"Conflicting ground-truth files: {candidates}"
            )

    GT_FILES[seq] = candidates[0]

    frames = sorted(
        (SEQ_DIRS[seq] / "img1").glob("*.jpg"),
        key=lambda p: int(p.stem),
    )

    frame_numbers = [int(p.stem) for p in frames]

    if frame_numbers != list(range(1, len(frames) + 1)):
        raise RuntimeError(
            f"{seq} has missing or non-consecutive frames."
        )

    info_path = SEQ_DIRS[seq] / "seqinfo.ini"

    if not info_path.exists():
        info_path = GT_FILES[seq].parent.parent / "seqinfo.ini"

    info = configparser.ConfigParser()
    info.read(info_path)

    if not info.has_section("Sequence"):
        raise FileNotFoundError(
            f"Cannot read seqinfo.ini for {seq}"
        )

    FRAME_PATHS[seq] = frames
    SEQ_LENGTHS[seq] = len(frames)
    SEQ_FPS[seq] = float(info["Sequence"]["frameRate"])

# ------------------------------------------------------------
# Verify tracker construction and record the configuration.
# ------------------------------------------------------------

probe_tracker = make_tracker()
config_object = getattr(probe_tracker, "config", None)

if hasattr(config_object, "to_dict"):
    resolved_config = config_object.to_dict()
else:
    resolved_config = {
        "api": TRACKER_API,
        "constructor_parameters": (
            {
                "reid_weights": str(REID_WEIGHTS),
                "device": "cuda:0",
                "half": False,
                **(
                    {"use_embeddings": True}
                    if "use_embeddings" in legacy_parameters else {}
                ),
                **(
                    {"per_class": False}
                    if "per_class" in legacy_parameters else {}
                ),
            }
            if TRACKER_API == "legacy_reid_weights"
            else {"reid_model": REID_MODEL}
        ),
        "other_parameters": "Installed implementation defaults",
    }

del probe_tracker
gc.collect()
torch.cuda.empty_cache()

manifest = {
    "ultralytics_version": version("ultralytics"),
    "boxmot_version": version("boxmot"),
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(DEVICE),
    "models": MODELS,
    "tracker": "boosttrack",
    "tracker_api": TRACKER_API,
    "tracker_config": resolved_config,
    "reid_model": (
        str(REID_WEIGHTS)
        if TRACKER_API == "legacy_reid_weights"
        else REID_MODEL
    ),
    "reid_precision": REID_PRECISION,
    "imgsz": IMGSZ,
    "conf": CONF,
    "iou": IOU,
    "max_det": MAX_DET,
    "detector_precision": PRECISION,
    "sequence_lengths": SEQ_LENGTHS,
}

(OUT / "configuration.json").write_text(
    json.dumps(manifest, indent=2, default=str)
)

print(json.dumps(manifest, indent=2, default=str))

for seq in SEQUENCES:
    print(
        f"{seq}: {SEQ_LENGTHS[seq]} frames, "
        f"{SEQ_FPS[seq]} FPS, GT: {GT_FILES[seq]}"
    )

print("\nCell 2 completed. Continue with Cell 3.")

# 

In [ ]:
def synchronize():
    torch.cuda.synchronize(DEVICE)

system_rows = []

for model_name, weights in MODELS.items():
    run_name = f"{model_name}_boosttrack"
    run_dir = OUT / "predictions" / run_name / "data"
    run_dir.mkdir(parents=True, exist_ok=True)

    for seq in SEQUENCES:
        gc.collect()
        torch.cuda.empty_cache()

        detector = YOLO(weights)
        tracker = make_tracker()
        frames = FRAME_PATHS[seq]

        common_args = {
            "classes": [0],        # COCO person
            "imgsz": IMGSZ,
            "conf": CONF,
            "iou": IOU,
            "max_det": MAX_DET,
            "device": DEVICE,
            "quantize": PRECISION,
            "verbose": False,
            "save": False,
        }

        first_frame = cv2.imread(str(frames[0]))
        if first_frame is None:
            raise RuntimeError(f"Cannot read {frames[0]}")

        # Warm up detection without advancing the tracker.
        for _ in range(WARMUP_FRAMES):
            warmup_result = detector.predict(
                first_frame, **common_args
            )
        del warmup_result
        synchronize()
        torch.cuda.reset_peak_memory_stats(DEVICE)

        timing_rows = []
        sequence_start = time.perf_counter()

        with (run_dir / f"{seq}.txt").open("w") as output:
            for frame_path in tqdm(
                frames, desc=f"{run_name} | {seq}"
            ):
                frame_number = int(frame_path.stem)

                synchronize()
                start = time.perf_counter()

                frame = cv2.imread(str(frame_path))
                if frame is None:
                    raise RuntimeError(f"Cannot read {frame_path}")

                result = detector.predict(
                    frame, **common_args
                )[0]

                if result.boxes is not None and len(result.boxes):
                    boxes = result.boxes
                    detections = np.column_stack([
                        boxes.xyxy.cpu().numpy(),
                        boxes.conf.cpu().numpy(),
                        boxes.cls.cpu().numpy(),
                    ]).astype(np.float32)
                else:
                    detections = np.empty((0, 6), dtype=np.float32)

                # Input:  N x 6 [x1,y1,x2,y2,score,class]
                # Output: M x 8 [x1,y1,x2,y2,id,score,class,det_index]
                tracks = tracker.update(detections, frame)

                synchronize()
                elapsed_ms = (time.perf_counter() - start) * 1000

                tracks = np.asarray(tracks)
                if tracks.ndim != 2 or tracks.shape[1] != 8:
                    raise RuntimeError(
                        f"Unexpected BoostTrack output: {tracks.shape}"
                    )

                timing_rows.append({
                    "frame": frame_number,
                    "latency_ms": elapsed_ms,
                    "detections": len(detections),
                    "active_tracks": len(tracks),
                })

                for track in tracks:
                    x1, y1, x2, y2 = map(float, track[:4])
                    track_id = int(track[4])
                    score = float(track[5])

                    output.write(
                        f"{frame_number},{track_id},"
                        f"{x1:.3f},{y1:.3f},"
                        f"{x2-x1:.3f},{y2-y1:.3f},"
                        f"{score:.6f},-1,-1,-1\n"
                    )

        wall_time_s = time.perf_counter() - sequence_start
        timings = pd.DataFrame(timing_rows)
        latency = timings["latency_ms"].to_numpy()

        # Exclude startup frame from steady-state speed statistics.
        steady = latency[1:]
        steady_time_s = float(steady.sum() / 1000)

        record = {
            "run": run_name,
            "model": model_name,
            "tracker": "boosttrack",
            "sequence": seq,
            "frames": len(frames),
            "steady_state_frames": len(steady),
            "fps": (
                len(steady) / steady_time_s
                if steady_time_s > 0 else np.nan
            ),
            "mean_latency_ms": (
                float(steady.mean()) if len(steady) else np.nan
            ),
            "p50_latency_ms": (
                float(np.percentile(steady, 50))
                if len(steady) else np.nan
            ),
            "p95_latency_ms": (
                float(np.percentile(steady, 95))
                if len(steady) else np.nan
            ),
            "first_frame_latency_ms": float(latency[0]),
            "processing_time_s": float(latency.sum() / 1000),
            "sequence_wall_time_s": wall_time_s,
            "peak_gpu_allocated_mb": (
                torch.cuda.max_memory_allocated(DEVICE) / 1024**2
            ),
            "peak_gpu_reserved_mb": (
                torch.cuda.max_memory_reserved(DEVICE) / 1024**2
            ),
        }

        system_rows.append(record)
        timings.to_csv(
            run_dir.parent / f"{seq}_timings.csv", index=False
        )
        pd.DataFrame(system_rows).to_csv(
            OUT / "system_per_sequence.csv", index=False
        )
        print(record)

        del detector, tracker, result, frame, first_frame
        gc.collect()
        torch.cuda.empty_cache()

system_df = pd.DataFrame(system_rows)
display(system_df.round(3))

In [ ]:
sys.path.insert(0, str(TRACKEVAL_PATH))
import trackeval

TE_GT = OUT / "trackeval_gt"
TE_GT.mkdir(exist_ok=True)

for seq in SEQUENCES:
    target = TE_GT / seq / "gt"
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(GT_FILES[seq], target / "gt.txt")

RUN_NAMES = [f"{model}_boosttrack" for model in MODELS]

eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({
    "USE_PARALLEL": False,
    "PRINT_RESULTS": True,
    "PRINT_CONFIG": False,
    "OUTPUT_SUMMARY": True,
    "OUTPUT_DETAILED": True,
    "PLOT_CURVES": False,
    "BREAK_ON_ERROR": True,
})

dataset_config = (
    trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
)
dataset_config.update({
    "GT_FOLDER": str(TE_GT),
    "TRACKERS_FOLDER": str(OUT / "predictions"),
    "OUTPUT_FOLDER": str(OUT / "tracking_evaluation"),
    "TRACKERS_TO_EVAL": RUN_NAMES,
    "CLASSES_TO_EVAL": ["pedestrian"],
    "BENCHMARK": "MOT20",
    "SPLIT_TO_EVAL": "train",
    "SKIP_SPLIT_FOL": True,
    "TRACKER_SUB_FOLDER": "data",
    "SEQ_INFO": SEQ_LENGTHS,
    "GT_LOC_FORMAT": "{gt_folder}/{seq}/gt/gt.txt",
    "DO_PREPROC": True,
    "PRINT_CONFIG": False,
})

dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
metrics = [
    trackeval.metrics.HOTA(),
    trackeval.metrics.CLEAR(),
    trackeval.metrics.Identity(),
]

evaluation_results, evaluation_messages = (
    trackeval.Evaluator(eval_config).evaluate([dataset], metrics)
)

tracking_rows = []

for run_name in RUN_NAMES:
    combined = evaluation_results[dataset.get_name()][run_name][
        "COMBINED_SEQ"
    ]["pedestrian"]

    hota = combined["HOTA"]
    clear = combined["CLEAR"]
    identity = combined["Identity"]

    tracking_rows.append({
        "run": run_name,
        "HOTA": float(np.mean(hota["HOTA"]) * 100),
        "IDF1": float(identity["IDF1"] * 100),
        "MOTA": float(clear["MOTA"] * 100),
        "ID_switches": int(clear["IDSW"]),
        "fragmentations": int(clear["Frag"]),
        "false_positives": int(clear["CLR_FP"]),
        "false_negatives": int(clear["CLR_FN"]),
        "tracking_precision": float(clear["CLR_Pr"] * 100),
        "tracking_recall": float(clear["CLR_Re"] * 100),
    })

tracking_df = pd.DataFrame(tracking_rows)
tracking_df.to_csv(OUT / "tracking_summary.csv", index=False)
display(tracking_df.round(3))

In [ ]:
system_df = pd.read_csv(OUT / "system_per_sequence.csv")
tracking_df = pd.read_csv(OUT / "tracking_summary.csv")

system_summary_rows = []

for run_name in RUN_NAMES:
    subset = system_df[system_df["run"] == run_name]

    # Exclude the startup frame of each sequence for speed statistics.
    latency = np.concatenate([
        pd.read_csv(
            OUT / "predictions" / run_name / f"{seq}_timings.csv"
        )["latency_ms"].to_numpy()[1:]
        for seq in SEQUENCES
    ])

    system_summary_rows.append({
        "run": run_name,
        "steady_state_FPS": len(latency) / (latency.sum() / 1000),
        "mean_latency_ms": float(latency.mean()),
        "P50_latency_ms": float(np.percentile(latency, 50)),
        "P95_latency_ms": float(np.percentile(latency, 95)),
        "peak_GPU_allocated_MB": subset["peak_gpu_allocated_mb"].max(),
        "peak_GPU_reserved_MB": subset["peak_gpu_reserved_mb"].max(),
        "total_processing_s": subset["processing_time_s"].sum(),
        "total_sequence_wall_s": subset["sequence_wall_time_s"].sum(),
    })

summary = tracking_df.merge(
    pd.DataFrame(system_summary_rows), on="run"
)
summary["R@1"] = np.nan
summary["R@5"] = np.nan

summary.to_csv(OUT / "benchmark_summary.csv", index=False)
display(summary.round(3))
print("Saved:", OUT / "benchmark_summary.csv")

In [ ]:
from IPython.display import Video, display

VIS_RUN = "yolo26m_boosttrack"    # Or "yolo26s_boosttrack"
VIS_SEQ = "MOT20-05"
PREVIEW_FRAMES = 300             # None = full sequence
VIDEO_WIDTH = 1280

prediction_file = (
    OUT / "predictions" / VIS_RUN / "data" / f"{VIS_SEQ}.txt"
)

tracks_by_frame = {}
for line in prediction_file.read_text().splitlines():
    if line.strip():
        row = [float(v) for v in line.split(",")]
        tracks_by_frame.setdefault(int(row[0]), []).append(row)

paths = FRAME_PATHS[VIS_SEQ]
if PREVIEW_FRAMES is not None:
    paths = paths[:PREVIEW_FRAMES]
if not paths:
    raise ValueError("No frames selected.")

first = cv2.imread(str(paths[0]))
if first is None:
    raise RuntimeError(f"Cannot read {paths[0]}")

height, width = first.shape[:2]
video_width = VIDEO_WIDTH - VIDEO_WIDTH % 2
video_height = int(height * video_width / width)
video_height -= video_height % 2
scale_x = video_width / width
scale_y = video_height / height

video_dir = OUT / "videos"
video_dir.mkdir(exist_ok=True)

raw_video = video_dir / f"{VIS_RUN}_{VIS_SEQ}_raw.mp4"
final_video = video_dir / f"{VIS_RUN}_{VIS_SEQ}.mp4"

writer = cv2.VideoWriter(
    str(raw_video),
    cv2.VideoWriter_fourcc(*"mp4v"),
    SEQ_FPS[VIS_SEQ],
    (video_width, video_height),
)
if not writer.isOpened():
    raise RuntimeError("Could not open video writer.")

try:
    for path in tqdm(paths, desc="Rendering BoostTrack video"):
        frame = cv2.imread(str(path))
        if frame is None:
            raise RuntimeError(f"Cannot read {path}")

        frame = cv2.resize(frame, (video_width, video_height))
        frame_number = int(path.stem)
        rows = tracks_by_frame.get(frame_number, [])

        for row in rows:
            track_id = int(row[1])
            x, y, w, h = row[2:6]
            color = tuple(
                int(v)
                for v in np.random.default_rng(track_id).integers(
                    70, 256, size=3
                )
            )

            p1 = (round(x * scale_x), round(y * scale_y))
            p2 = (round((x + w) * scale_x), round((y + h) * scale_y))

            cv2.rectangle(frame, p1, p2, color, 2)
            cv2.putText(
                frame,
                f"ID {track_id} | {row[6]:.2f}",
                (max(0, p1[0]), max(20, p1[1] - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5, color, 2, cv2.LINE_AA,
            )

        cv2.putText(
            frame,
            f"{VIS_RUN} | frame {frame_number} | {len(rows)} tracks",
            (15, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (255, 255, 255), 2, cv2.LINE_AA,
        )
        writer.write(frame)
finally:
    writer.release()

if shutil.which("ffmpeg"):
    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(raw_video),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(final_video),
        ],
        check=True,
    )
    display(Video(str(final_video), embed=True, width=960))
    print("Saved:", final_video)
else:
    print("Download from Kaggle Output:", raw_video)